In [1]:
import torch
import torch.nn as nn
import random
from typing import List, Tuple, Optional

In [2]:
xq = torch.randn(1, 1892, 40, 128) # bsz, seqlen, self.n_local_heads, self.head_dim
xk = torch.randn(1, 1892, 40, 128)

In [3]:
def precompute_freqs_cis(
  dim: int,
  end: int,
  theta: float = 10000.0,
  device: Optional[torch.device] = None,
) -> torch.Tensor:
  device = device or torch.device("cpu")
  freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device)
                           [: (dim // 2)].float() / dim))
  t = torch.arange(end, device=device)  # type: ignore
  freqs = torch.outer(t, freqs).float()  # type: ignore
  freqs_cis = torch.cat([freqs, freqs], dim=-1)
  # freqs_cis = freqs.reshape(*freqs.shape[:-2], -1)
  
  sin_freqs_cis = freqs_cis.sin()
  cos_freqs_cis = freqs_cis.cos()

  # (seq_len, dim // 2, 2)
  freqs_cis = torch.stack([sin_freqs_cis, cos_freqs_cis], dim=-1)
  # (seq_len, n_heads, dim // 2, 2)
  freqs_cis = freqs_cis[:, None, :, :]
  return freqs_cis

def rotate_half_neg(x: torch.Tensor):
  # ndim = x.ndim
  fdim = x.shape[-1]
  x2 = x[..., :fdim // 2]
  x1 = x[..., fdim // 2:]
  y = torch.cat([-x2, x1], dim=-1)
  # y = y.flatten(ndim - 1)
  return y

def cos_sin_apply_rotary_emb(
  xq: torch.Tensor,
  xk: torch.Tensor,
  freqs_cis: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
  sin_freqs_cis = freqs_cis[..., 0]
  cos_freqs_cis = freqs_cis[..., 1]
  xqe = (xq * cos_freqs_cis) + (rotate_half_neg(xq) * sin_freqs_cis)
  xke = (xk * cos_freqs_cis) + (rotate_half_neg(xk) * sin_freqs_cis)
  return xqe.type_as(xq), xke.type_as(xk)


freqs_cis = precompute_freqs_cis(
                768 // 12,
                16,
                10_000,
                torch.device("cpu"),
              )

xq_res, xk_res = cos_sin_apply_rotary_emb(xq, xk, freqs_cis)


In [4]:

class LlamaRotaryEmbedding(torch.nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()

        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2).float().to(device) / self.dim))
        self.register_buffer("inv_freq", inv_freq)

        # Build here to make `torch.jit.trace` work.
        self._set_cos_sin_cache(
            seq_len=max_position_embeddings, device=self.inv_freq.device, dtype=torch.get_default_dtype()
        )
    def forward(self, x, seq_len=None):
        # x: [bs, num_attention_heads, seq_len, head_size]
        if seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(seq_len=seq_len, device=x.device, dtype=x.dtype)

        return (
            self.cos_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
        )
    
class LlamaLinearScalingRotaryEmbedding(LlamaRotaryEmbedding):
    """LlamaRotaryEmbedding extended with linear scaling. Credits to the Reddit user /u/kaiokendev"""

    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None, scaling_factor=1.0):
        self.scaling_factor = scaling_factor
        super().__init__(dim, max_position_embeddings, base, device)

    def _set_cos_sin_cache(self, seq_len, device, dtype):
        self.max_seq_len_cached = seq_len
        t = torch.arange(self.max_seq_len_cached, device=device, dtype=self.inv_freq.dtype)
        t = t / self.scaling_factor

        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        # Different from paper, but it uses a different permutation in order to obtain the same calculation
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :].to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :].to(dtype), persistent=False)

    def forward(self, x, seq_len=None):
        # x: [bs, num_attention_heads, seq_len, head_size]
        if seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(seq_len=seq_len, device=x.device, dtype=x.dtype)

        return (
            self.cos_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
        )

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, position_ids):
    # The first two dimensions of cos and sin are always 1, so we can `squeeze` them.
    cos = cos.squeeze(1).squeeze(0)  # [seq_len, dim]
    sin = sin.squeeze(1).squeeze(0)  # [seq_len, dim]
    cos = cos[position_ids].unsqueeze(1)  # [bs, 1, seq_len, dim]
    sin = sin[position_ids].unsqueeze(1)  # [bs, 1, seq_len, dim]
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

rot_emb_hf = LlamaLinearScalingRotaryEmbedding(768 // 12, 16, 10_000, torch.device("cpu"))
cos, sin = rot_emb_hf(xq, 16)
query_states, key_states = apply_rotary_pos_emb(xq, xk, cos, sin, torch.arange(16).unsqueeze(0).unsqueeze(0))

In [4]:
def precompute_freqs_cis_og(dim: int, end: int, theta: float = 10000.0):
  freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)
                 [: (dim // 2)].float() / dim))
  t = torch.arange(end, device=freqs.device)  # type: ignore
  freqs = torch.outer(t, freqs).float()  # type: ignore
  freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
  return freqs_cis


def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
  ndim = x.ndim
  # assert 0 <= 1 < ndim
  # assert freqs_cis.shape == (x.shape[1], x.shape[-1]), f"{freqs_cis.shape} != {(x.shape[1], x.shape[-1])}"
  shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
  return freqs_cis.view(*shape)


def apply_rotary_emb_og(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
  xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
  xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
  freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
  xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
  xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
  return xq_out.type_as(xq), xk_out.type_as(xk)


freqs_cis_og = precompute_freqs_cis_og(128, 1892, 10_000)

xq_og, xk_og = apply_rotary_emb_og(xq, xk, freqs_cis_og)

In [5]:
def precompute_freqs_cis_new(
  dim: int,
  end: int,
  theta: float = 10000.0,
  device: Optional[torch.device] = None,
) -> torch.Tensor:
  device = device or torch.device("cpu")
  freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device)
                           [: (dim // 2)].float() / dim))
  t = torch.arange(end, device=device)  # type: ignore
  freqs = torch.outer(t, freqs).float()  # type: ignore
  freqs_sin = torch.sin(freqs)
  freqs_cos = torch.cos(freqs)
  freqs_cis = torch.stack([freqs_sin, freqs_cos], dim=-1)
  return freqs_cis

def reshape_for_broadcast_new(freqs_cis: torch.Tensor, x: torch.Tensor):
  ndim = x.ndim
  shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
  return freqs_cis.view(*shape)

def _rotate(x: torch.Tensor, sin: torch.Tensor, cos: torch.Tensor) -> torch.Tensor:
  x_real = x[..., 0::2]
  x_imag = x[..., 1::2]

  freq_cos = reshape_for_broadcast_new(cos, x_real).float()
  freq_sin = reshape_for_broadcast_new(sin, x_imag).float()

  x_out_real = x_real * freq_cos - x_imag * freq_sin
  x_out_imag = x_real * freq_sin + x_imag * freq_cos
  x_out = torch.stack([x_out_real, x_out_imag], dim=-1).flatten(-2)
  return x_out.type_as(x)

def cos_sin_apply_rotary_emb_new(
  xq: torch.Tensor,
  xk: torch.Tensor,
  freqs_cis: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
  sin, cos = freqs_cis[..., 0], freqs_cis[..., 1]
  return _rotate(xq, sin, cos), _rotate(xk, sin, cos)


freqs_cis_new = precompute_freqs_cis_new(
                128,
                1892,
                10_000,
              )

xq_new, xk_new = cos_sin_apply_rotary_emb_new(xq, xk, freqs_cis_new)

In [15]:
# compare with llama
print(torch.allclose(query_states, xq_res, atol=1e-6), torch.allclose(key_states, xk_res, atol=1e-5))
print(torch.allclose(query_states, xq_og, atol=1e-6), torch.allclose(key_states, xk_og, atol=1e-5))
print(torch.allclose(query_states, xq_new, atol=1e-6), torch.allclose(key_states, xk_new, atol=1e-5))

False False
False False
False False


In [20]:
rand_idxc = random.randint(0, 15)
xq_res[0][rand_idxc][3][:10], xq_og[0][rand_idxc][3][:10], xq_new[0][rand_idxc][3][:10]

(tensor([ 0.1081,  0.4202,  0.0149,  0.7793,  0.1034, -1.8379, -0.0909,  0.0765,
         -0.1083, -0.4319]),
 tensor([ 0.6394,  0.9638,  0.0020, -1.3049,  1.2606, -1.1337, -0.0733,  0.3874,
          1.4873, -0.0078]),
 tensor([ 0.6394,  0.9638,  0.0020, -1.3049,  1.2606, -1.1337, -0.0733,  0.3874,
          1.4873, -0.0078]))

In [15]:
torch.allclose(xq_new, xq_og, atol=1e-6), torch.allclose(xk_new, xk_og, atol=1e-6)

(True, True)